In [ ]:
import gym
import torch
import torch.nn as nn
import torch.multiprocessing as mp
import numpy as np

class RouteEnv(gym.Env):
    def __init__(self):
        self.observation_space = spaces.Box(low=0, high=10, shape=(2,))
        self.action_space = spaces.Discrete(4)  # بالا، پایین، چپ، راست
        self.pos = [0, 0]

    def reset(self):
        self.pos = [0, 0]
        return np.array(self.pos)

    def step(self, action):
        if action == 0: self.pos[0] += 1
        if action == 1: self.pos[0] -= 1
        if action == 2: self.pos[1] += 1
        if action == 3: self.pos[1] -= 1
        self.pos = np.clip(self.pos, 0, 10)
        dist = -np.sum((np.array(self.pos) - np.array([10, 10]))**2)
        done = np.array_equal(self.pos, [10, 10])
        return np.array(self.pos), dist, done, {}

class A3CNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.shared = nn.Linear(input_dim, 32)
        self.policy = nn.Linear(32, output_dim)
        self.value = nn.Linear(32, 1)

    def forward(self, x):
        h = torch.relu(self.shared(x))
        logits = self.policy(h)
        value = self.value(h)
        return logits, value

def worker(name, global_net, optimizer):
    env = RouteEnv()
    local_net = A3CNet(2, 4)
    state = env.reset()
    while True:
        hx = torch.FloatTensor(state)
        for step in range(5):
            logits, value = local_net(hx)
            probs = torch.softmax(logits, dim=-1)
            action = probs.multinomial(1).item()
            next_state, reward, done, _ = env.step(action)
            if done:
                next_state = env.reset()
            state = next_state
            hx = torch.FloatTensor(next_state)

# این بخش برای اجرا در محیط‌های پشتیبان از multiprocessing مناسب است